In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.features import add_features

df = pd.read_csv("../data/raw/fraudTrain.csv", index_col=0,
                 parse_dates=["trans_date_trans_time"])

# Two columns claim to hold the same time. Do they agree?
print(pd.to_datetime(df["unix_time"].iloc[0], unit="s"), "vs", df["trans_date_trans_time"].iloc[0])

df = add_features(df)
cutoff = pd.Timestamp("2020-01-01")
tr  = df[df["trans_date_trans_time"] <  cutoff].copy()
val = df[df["trans_date_trans_time"] >= cutoff].copy()

2012-01-01 00:00:18 vs 2019-01-01 00:00:18


In [2]:
new_cols = ["hours_since_prev", "amt_vs_card_avg", "txn_count_1h",
            "txn_count_24h", "amt_sum_24h", "distance_km"]
print(tr.groupby("is_fraud")[new_cols].agg(["median", "mean"]).round(2).T)

is_fraud                      0        1
hours_since_prev median    4.37     1.33
                 mean      8.60     5.82
amt_vs_card_avg  median    0.66     5.03
                 mean      0.99     6.80
txn_count_1h     median    0.00     0.00
                 mean      0.20     0.67
txn_count_24h    median    3.00     4.00
                 mean      4.10     4.30
amt_sum_24h      median  181.46  1686.86
                 mean    280.06  1950.47
distance_km      median   78.24    78.21
                 mean     76.12    76.53


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score

skewed = ["amt", "hours_since_prev", "amt_vs_card_avg",
          "txn_count_1h", "amt_sum_1h", "txn_count_24h", "amt_sum_24h"]
plain  = ["distance_km"]
cats   = ["category", "hour"]
features = skewed + plain + cats

preprocess = ColumnTransformer([
    ("skewed", Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("log", FunctionTransformer(np.log1p)),
                         ("scale", StandardScaler())]), skewed),
    ("plain", StandardScaler(), plain),
    ("cats", OneHotEncoder(handle_unknown="ignore"), cats),
])
model = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])
model.fit(tr[features], tr["is_fraud"])
val_scores = model.predict_proba(val[features])[:, 1]

print(f"PR-AUC: {average_precision_score(val['is_fraud'], val_scores):.3f}  (baseline 0.438)")
for t in [0.5, 0.3, 0.1, 0.05]:
    pred = (val_scores >= t).astype(int)
    print(f"threshold {t:.2f}: precision {precision_score(val['is_fraud'], pred):.3f} | "
          f"recall {recall_score(val['is_fraud'], pred):.3f} | flags {pred.sum():,}")

PR-AUC: 0.697  (baseline 0.438)
threshold 0.50: precision 0.871 | recall 0.506 | flags 1,329
threshold 0.30: precision 0.785 | recall 0.594 | flags 1,729
threshold 0.10: precision 0.504 | recall 0.725 | flags 3,287
threshold 0.05: precision 0.332 | recall 0.791 | flags 5,453
